# Image Classifier: Train and Deploy with Gradio
### An annotated walkthrough

This notebook trains a cat vs dog image classifier and deploys it as an interactive web app.
Every key decision is explained along the way.

**Before running:** Go to Runtime → Change runtime type → T4 GPU

---

### The library stack

Three libraries do the work here:

| Library | Role |
|---------|------|
| **PyTorch** | The core deep learning framework, developed by Meta. Does all the mathematical heavy lifting — tensor operations, automatic differentiation, GPU computation. |
| **fastai** | A high-level wrapper around PyTorch, developed to accompany the fast.ai course. Provides user-friendly functions like `vision_learner` that would otherwise require hundreds of lines of PyTorch code. |
| **fastbook** | The companion package to the fast.ai textbook. Imports fastai and a collection of other useful tools in one line. |

When you call `from fastbook import *`, you are pulling in fastai, which in turn sits on top of PyTorch.

## Step 1: Install libraries

In [ ]:
# fastbook installs fastai and PyTorch as dependencies
# gradio is the deployment framework
# -q suppresses verbose pip output
!pip install fastbook gradio -q

## Step 2: Load and prepare the data

We use fastai's built-in **Oxford-IIIT Pet Dataset** — 7,390 images of 37 cat and dog breeds.
Using a built-in dataset avoids any dependency on external image search APIs.

In [ ]:
from fastbook import *
from fastai.vision.all import *

# untar_data downloads and unpacks the dataset to a local cache folder.
# URLs.PETS is a shortcut constant pointing to the Oxford Pets dataset URL.
path = untar_data(URLs.PETS)

# The Oxford Pets dataset uses a naming convention where cat breeds start
# with an uppercase letter and dog breeds start with a lowercase letter.
# This function exploits that convention to assign labels without needing
# a separate label file.
def is_cat(filename):
    return filename[0].isupper()

# ImageDataLoaders is a fastai class that handles the full data pipeline:
#   - Locating all image files
#   - Assigning labels (via label_func)
#   - Splitting into training and validation sets (valid_pct=0.2 = 20% validation)
#   - Resizing images to 224x224 pixels (required by ResNet)
#   - Batching images for efficient GPU processing
#   - Applying data augmentation (random flips, crops) to improve generalisation
# seed=42 ensures the train/validation split is reproducible across runs.
dls = ImageDataLoaders.from_name_func(
    path,
    get_image_files(path/"images"),
    valid_pct=0.2,
    seed=42,
    label_func=is_cat,
    item_tfms=Resize(224)
)

# Preview a random sample of images with their labels
dls.show_batch(max_n=6)

## Step 3: Train the model

### Transfer learning

Training a deep learning model from scratch requires millions of images and days of compute time.
**Transfer learning** sidesteps this by starting from a model that has already been trained on a large dataset,
then adapting it to a new, specific task.

Here we use **ResNet34** as our starting point:
- Developed by Microsoft Research in 2015
- 34 layers deep
- Pretrained on **ImageNet** — 1.2 million images across 1,000 categories
- Already understands edges, textures, shapes, and objects from that training

We adapt it to our cat/dog task by replacing only the final layer with one that outputs two classes,
then training on our dataset.

### What `vision_learner` does

`vision_learner(dls, resnet34, metrics=error_rate)` does several things automatically:
1. Downloads the pretrained ResNet34 weights
2. Removes the original ImageNet output layer (1,000 classes)
3. Replaces it with a new output layer for our 2 classes (Cat / Dog)
4. Sets up the training loop, optimiser, and loss function

### What `fine_tune` does

`fine_tune(3)` runs transfer learning in two stages automatically:
1. **Freeze** — trains only the new output layer for one epoch, leaving the pretrained weights untouched
2. **Unfreeze** — trains the entire network for the remaining epochs, using different learning rates for different layers (discriminative learning rates)

The `3` is the number of epochs — complete passes through the training data.
Each epoch the model sees all training images and updates its weights to reduce errors.

In [ ]:
# vision_learner is a fastai function that creates a convolutional neural network
# for image classification using a pretrained architecture.
# Under the hood it is constructing a PyTorch model.
#
# Arguments:
#   dls       - the data loaders created in Step 2
#   resnet34  - the pretrained architecture to use as a base
#   metrics   - what to display during training (error_rate = % of wrong predictions)
learn = vision_learner(dls, resnet34, metrics=error_rate)

# learn.summary() prints every layer in the network, its output shape, and how many
# parameters it has. The pretrained ResNet34 layers are frozen at this point —
# only the new classification head (last few rows) has trainable parameters.
learn.summary()

# fine_tune runs transfer learning:
#   - First freezes the pretrained layers and trains only the new head
#   - Then unfreezes everything and trains the full network
# The argument (3) is the number of epochs for the unfrozen phase.
learn.fine_tune(3)

### Reading the training output

| Column | Meaning |
|--------|---------|
| `epoch` | Which pass through the data we are on |
| `train_loss` | How wrong the model is on training data — should decrease each epoch |
| `valid_loss` | How wrong the model is on validation data (images it has never seen during training) — the honest measure of performance |
| `error_rate` | Percentage of validation images classified incorrectly |

If `valid_loss` starts increasing while `train_loss` keeps falling, the model is **overfitting** —
memorising the training data rather than learning general patterns.

## Step 4: Evaluate the model

A **confusion matrix** shows where the model makes mistakes.
The diagonal represents correct predictions; off-diagonal entries are errors.
For our binary classifier this is a 2×2 matrix — for a 37-breed classifier it would be 37×37.

In [ ]:
# ClassificationInterpretation analyses the model's predictions on the validation set
interp = ClassificationInterpretation.from_learner(learn)

# plot_confusion_matrix visualises correct vs incorrect predictions
interp.plot_confusion_matrix()

## Step 4b: Visualise what the model learned (GradCAM)

**GradCAM** (Gradient-weighted Class Activation Mapping) produces a heatmap showing which pixels drove the model's prediction.

- **Green** areas = strong positive activation — the model is paying attention here
- **Red** areas = weak or negative activation — the model is ignoring this region

A well-trained model should highlight the animal itself (face, body) rather than the background.
This is a useful sanity check: if the heatmap focuses on the grass or sofa instead of the cat, the model may be learning spurious correlations rather than genuine features.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

activation = {}
gradient = {}

def fwd_hook(module, input, output):
    activation['last_conv'] = output

def bwd_hook(module, grad_input, grad_output):
    gradient['last_conv'] = grad_output[0]

last_conv = learn.model[0][-1][-1]
fwd_handle = last_conv.register_forward_hook(fwd_hook)
bwd_handle = last_conv.register_full_backward_hook(bwd_hook)

x, y = dls.valid.one_batch()
x, y = x[:4], y[:4]

# fastai wraps tensors in custom subclasses (TensorImage, TensorCategory).
# Strip them to plain torch.Tensor so standard PyTorch indexing works correctly.
x = x.as_subclass(torch.Tensor)
y = y.as_subclass(torch.Tensor)

learn.model.eval()
preds = learn.model(x)
preds[range(4), y].sum().backward()

fwd_handle.remove()
bwd_handle.remove()

acts = activation['last_conv']
grads = gradient['last_conv']
w = grads.mean(dim=[2, 3], keepdim=True)
cam_map = F.relu((w * acts).sum(1)).detach().cpu()

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    img = x[i].cpu()
    img_show = img.permute(1, 2, 0)
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())

    axes[0, i].imshow(img_show)
    axes[0, i].set_title(dls.vocab[y[i].item()])
    axes[0, i].axis('off')

    cam = cam_map[i]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    axes[1, i].imshow(img_show)
    axes[1, i].imshow(cam, alpha=0.6, cmap='RdYlGn',
                      extent=[0, 224, 224, 0], interpolation='bilinear')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('GradCAM overlay', fontsize=12)
plt.suptitle('Green = strong activation | Red = weak activation', fontsize=13)
plt.tight_layout()
plt.show()

## Step 5: Save the model

`learn.export()` saves the trained model to a `.pkl` file (Python pickle format).
This file contains both the model architecture and the learned weights,
so it can be loaded later without retraining.

We save to `/content/` — Colab's working directory — so the file is easy to find.

In [ ]:
learn.export('/content/cat_dog_classifier.pkl')
print('Model saved.')

## Step 6: Deploy with Gradio

**Gradio** is a Python library that wraps any function in a web interface.
You define what goes in (an image) and what comes out (a label), and Gradio handles the rest.

`share=True` generates a public URL valid for one week that anyone can open in a browser —
no server, no hosting, no account required.

In [ ]:
import gradio as gr
from fastai.vision.all import *
from datetime import datetime

# load_learner reads the saved .pkl file and reconstructs the model
learn = load_learner('/content/cat_dog_classifier.pkl')

# Timestamp generated at the moment this cell is run
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# This function is what Gradio calls each time a user submits an image.
# learn.predict() returns three values:
#   pred  - the predicted class label ('True' for cat, 'False' for dog)
#   idx   - the index of the predicted class
#   probs - a tensor of probabilities for each class
def classify_image(image):
    pred, idx, probs = learn.predict(image)
    label = 'Cat' if bool(pred) else 'Dog'
    confidence = probs[idx].item()
    return f"{label} ({confidence:.1%} confidence)"

# gr.Interface wraps the function in a web UI:
#   fn          - the function to call on each submission
#   inputs      - the type of input widget
#   outputs     - the type of output widget
app = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(),
    outputs=gr.Text(label="Prediction"),
    title="Cat or Dog Classifier",
    description=f"Upload a photo and the model will predict whether it is a cat or a dog.\n\nDeployed: {timestamp}"
)

app.launch(share=True)